# DBSCAN — Fitting & Evaluation (Minimal)

Notebook ini hanya menjalankan langkah fitting (final clustering) dan evaluasi.
Edit `INPUT_FILE`, `NORMALIZE`, dan `CHOSEN_EPS/CHOSEN_MIN_SAMPLES` sesuai kebutuhan sebelum menjalankan.

In [ ]:
# Imports & GPU detection
import time
import gc
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Try GPU (cuML) — fallback to sklearn if not available
USE_GPU = False
try:
    import cupy as cp
    from cuml.cluster import DBSCAN as cuDBSCAN
    from cuml.neighbors import NearestNeighbors as cuNearestNeighbors
    USE_GPU = True
except Exception:
    USE_GPU = False

# sklearn fallback imports
from sklearn.cluster import DBSCAN as skDBSCAN
from sklearn.neighbors import NearestNeighbors as skNearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import normalize as sklearn_normalize

print('USE_GPU =', USE_GPU)

In [ ]:
# Part: Smart loaders & normalization (aligned with dbscan_experiments_gpu)
from pathlib import Path

def detect_embedding_dim(file_path: Path) -> int:
    """Auto-detect embedding dimension from filename."""
    filename = file_path.name.lower()
    if 'pca256' in filename:
        return 256
    elif 'pca128' in filename:
        return 128
    else:
        return 768


def infer_num_rows(path: Path, embedding_dim: int = None) -> int:
    """Infer number of rows for RAW float32 memmap files."""
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim(path)
    size = path.stat().st_size
    return size // (embedding_dim * np.dtype(np.float32).itemsize)


def load_single_file_smart(file_path: Path, embedding_dim: int = None):
    """
    Smart loader (same approach as dbscan_experiments_gpu):
    1) Try regular .npy via np.load(mmap_mode='r')
    2) If that fails, treat file as RAW float32 memmap with inferred rows
    """
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim(file_path)
        print(f"   🔍 Auto-detected dimension: {embedding_dim}")

    try:
        arr = np.load(file_path, mmap_mode='r')
        detected_dim = arr.shape[1]
        if detected_dim != embedding_dim:
            print(f"   ⚠️ Dimension mismatch! Expected {embedding_dim}, got {detected_dim}")
            embedding_dim = detected_dim
        return arr, True, arr.shape[0]
    except Exception as e:
        # Fallback: assume RAW float32 binary with shape (num_rows, embedding_dim)
        num_rows = infer_num_rows(file_path, embedding_dim)
        arr = np.memmap(file_path, dtype=np.float32, mode='r', shape=(num_rows, embedding_dim))
        print(f"   ⚠️ Loaded as RAW memmap: {num_rows:,} rows × {embedding_dim} dims (np.load failed: {type(e).__name__})")
        return arr, True, num_rows


def load_embeddings_from_files(files):
    """Load embeddings with auto-dimension detection."""
    if len(files) == 0:
        raise FileNotFoundError('No embedding files provided')

    for f in files:
        if not f.exists():
            raise FileNotFoundError(f'File not found: {f}')

    first_arr, _, _ = load_single_file_smart(files[0])
    embedding_dim = first_arr.shape[1]
    print(f"Embedding dimension: {embedding_dim}")

    if len(files) == 1:
        print(f"Loading single file: {files[0].name}")
        return first_arr

    print(f"Loading {len(files)} files...")
    total_samples = 0
    file_info = []
    for f in files:
        arr, is_mmap, n_rows = load_single_file_smart(f)
        file_info.append((f, arr, n_rows))
        total_samples += n_rows
        print(f"  - {f.name}: {n_rows:,} rows")

    total_size_gb = (total_samples * embedding_dim * 4) / (1024**3)
    print(f"\nTotal samples: {total_samples:,} ({total_size_gb:.2f} GB)")

    if total_size_gb < 100:
        arrays = [arr for _, arr, _ in file_info]
        return np.vstack(arrays)
    else:
        raise MemoryError(f"Dataset too large ({total_size_gb:.1f}GB)")


def normalize_large_array_batched(arr, norm='l2', batch_size=1000000, size_threshold_gb=15):
    """
    Normalization logic aligned with the original notebook (without tqdm dependency).
    """
    n_samples, n_features = arr.shape
    size_gb = arr.nbytes / (1024**3)
    is_memmap = isinstance(arr, np.memmap)

    print(f"   Array: {n_samples:,} × {n_features} ({size_gb:.2f} GB)")
    print(f"   Type: {'memmap (read-only)' if is_memmap else 'in-memory array'}")

    # Prefer the same tmp location as the original notebook when available
    preferred_tmp = Path("/media/bioinfo04/Expansion/tmp")
    tmp_dir = preferred_tmp if preferred_tmp.exists() else Path("./tmp_norm")
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # CRITICAL: For large memmap (>10GB), NEVER load to RAM
    if is_memmap and size_gb > 10:
        print(f"   → Large memmap detected ({size_gb:.1f} GB)")
        print(f"   → Using batch normalization (memory-safe)")
        normalized_path = tmp_dir / f"normalized_embeddings_{np.random.randint(100000)}.npy"
        normalized = np.memmap(normalized_path, dtype=np.float32, mode='w+', shape=(n_samples, n_features))
        n_batches = (n_samples + batch_size - 1) // batch_size
        print(f"   → Processing {n_batches} batches of {batch_size:,} rows...")
        for i in range(n_batches):
            start_idx = i * batch_size
            end_idx = min((i + 1) * batch_size, n_samples)
            batch = arr[start_idx:end_idx].copy()
            norms = np.linalg.norm(batch, axis=1, keepdims=True)
            norms[norms == 0] = 1
            batch /= norms
            normalized[start_idx:end_idx] = batch
            del batch, norms
            if (i + 1) % 10 == 0:
                normalized.flush()
                print(f"     - normalized batches: {i+1}/{n_batches}")
        normalized.flush()
        print(f"   ✅ Normalized memmap created: {normalized_path}")
        return normalized

    # Small in-memory arrays: use sklearn normalize (fast)
    if (not is_memmap) and size_gb < size_threshold_gb:
        print(f"   → Small in-memory array ({size_gb:.1f} GB < {size_threshold_gb} GB)")
        try:
            normalized = sklearn_normalize(arr, norm=norm, copy=False)
            print("   ✅ Normalized with sklearn")
            return normalized
        except MemoryError:
            print("   ⚠️ MemoryError with sklearn, falling back to batch processing...")

    # Small memmap (<10GB): safe-ish to load to memory for sklearn normalization
    if is_memmap and size_gb <= 10:
        print(f"   → Small memmap ({size_gb:.1f} GB ≤ 10 GB)")
        try:
            arr_copy = np.array(arr)
            normalized = sklearn_normalize(arr_copy, norm=norm, copy=False)
            del arr_copy
            print("   ✅ Normalized with sklearn")
            return normalized
        except MemoryError:
            print("   ⚠️ MemoryError, falling back to batch processing...")

    # Fallback: batch processing
    print("   → Using batch normalization (safe fallback)")
    if is_memmap:
        normalized_path = tmp_dir / f"normalized_embeddings_{np.random.randint(100000)}.npy"
        normalized = np.memmap(normalized_path, dtype=np.float32, mode='w+', shape=(n_samples, n_features))
    else:
        normalized = arr
    n_batches = (n_samples + batch_size - 1) // batch_size
    print(f"   → Processing {n_batches} batches...")
    for i in range(n_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, n_samples)
        if is_memmap and normalized is not arr:
            batch = arr[start_idx:end_idx].copy()
            norms = np.linalg.norm(batch, axis=1, keepdims=True)
            norms[norms == 0] = 1
            batch /= norms
            normalized[start_idx:end_idx] = batch
            del batch, norms
        else:
            batch = normalized[start_idx:end_idx]
            norms = np.linalg.norm(batch, axis=1, keepdims=True)
            norms[norms == 0] = 1
            batch /= norms
            normalized[start_idx:end_idx] = batch
        if is_memmap and (i + 1) % 10 == 0:
            normalized.flush()
            print(f"     - normalized batches: {i+1}/{n_batches}")
    if is_memmap and normalized is not arr:
        normalized.flush()
        print(f"   ✅ Batch normalized (RAM-safe) → {normalized_path}")
    return normalized

In [ ]:
# Configuration: set path(s) to your embeddings file(s) (npy/memmap)
# Use a list of files like in the original notebook
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_thunderbird_embeddings.npy"),
]
NORMALIZE = False  # Set True if you want L2 normalization (or set USE_COSINE_DISTANCE)
USE_COSINE_DISTANCE = False

print('Checking INPUT_FILES...')
for p in INPUT_FILES:
    print(' -', p)
    if not p.exists():
        raise FileNotFoundError(f'Input file not found: {p} — set correct path')

# Load embeddings using smart loader
print('\nLoading embeddings (smart loader)...')
emb = load_embeddings_from_files(INPUT_FILES)
print(f'Loaded embeddings shape: {getattr(emb, "shape", "unknown")}')

# Normalize if requested (or if using cosine distance)
if USE_COSINE_DISTANCE or NORMALIZE:
    print('\n🔄 Normalizing embeddings for cosine distance...')
    emb = normalize_large_array_batched(emb, norm='l2', batch_size=1000000, size_threshold_gb=15)
    print('✓ Normalized')

# Duplicate analysis (small sample)
print('\n🔍 Performing duplicate analysis (small sample)...')
n_total = emb.shape[0]
rng = np.random.RandomState(42)
DUP_CHECK_SAMPLE = min(20000, n_total)
if DUP_CHECK_SAMPLE < n_total:
    dup_idx = rng.choice(n_total, DUP_CHECK_SAMPLE, replace=False)
    emb_dup_check = emb[dup_idx]
else:
    emb_dup_check = emb

print(f'   Running unique() on {len(emb_dup_check):,} samples...')
unique_embeddings = np.unique(emb_dup_check, axis=0)
n_unique = len(unique_embeddings)
duplicate_ratio = 1.0 - (n_unique / len(emb_dup_check))
print(f'   Unique: {n_unique:,} / {len(emb_dup_check):,} ({(1-duplicate_ratio)*100:.1f}%), duplicates: {duplicate_ratio*100:.1f}%')

# Prepare for clustering: if large dataset, we'll use batch-fit + assignment (handled in clustering cell)
print('\nConfiguration ready — proceed to clustering cell')

gc.collect()


In [ ]:
# Final DBSCAN parameters (set to your chosen values)
CHOSEN_EPS = 0.740706
CHOSEN_MIN_SAMPLES = 30
USE_COSINE_DISTANCE = False  # Set True only if embeddings were normalized for cosine

In [ ]:
# Final clustering (single-pass or batch-assignment) — GPU if available, otherwise CPU fallback
n_total = emb.shape[0]
# Estimate emb bytes (works for memmap and ndarray)
itemsize = int(getattr(emb, 'dtype', np.dtype(np.float32)).itemsize)
emb_size_gb = (n_total * emb.shape[1] * itemsize) / (1024**3)
print(f'Total samples: {n_total:,}, dataset size: {emb_size_gb:.2f} GB')
labels_full = None
if USE_GPU:
    import cupy as cp
    gpu_props = cp.cuda.runtime.getDeviceProperties(0)
    available_vram_gb = gpu_props['totalGlobalMem'] / (1024**3)
    usable_vram_gb = available_vram_gb * 0.7
    max_samples_per_batch = int((usable_vram_gb * (1024**3)) / (emb.shape[1] * itemsize))
    print(f'GPU VRAM: {available_vram_gb:.1f} GB (usable: {usable_vram_gb:.1f} GB)')
    print(f'Max batch size (approx): {max_samples_per_batch:,} samples')
    if n_total <= max_samples_per_batch:
        print('\nGPU single-pass clustering — transferring to GPU...')
        start_time = time.time()
        emb_gpu = cp.asarray(emb, dtype=cp.float32)
        model = cuDBSCAN(eps=float(CHOSEN_EPS), min_samples=int(CHOSEN_MIN_SAMPLES))
        labels_gpu = model.fit_predict(emb_gpu)
        labels_full = cp.asnumpy(labels_gpu)
        elapsed = time.time() - start_time
        print(f'GPU DBSCAN completed in {elapsed/60:.2f} minutes')
        del emb_gpu, labels_gpu, model
        cp.get_default_memory_pool().free_all_blocks()
    else:
        print('\nGPU batch strategy: fit on representative sample, then assign via k-NN')
        # Conservative fit sample (to avoid OOM)
        MAX_FIT_SAMPLES_GPU = min(1000000, max_samples_per_batch // 3)
        n_batches = (n_total + max_samples_per_batch - 1) // max_samples_per_batch
        samples_per_batch_target = max(1, MAX_FIT_SAMPLES_GPU // n_batches)
        print(f'Max fit samples (memory-safe): {MAX_FIT_SAMPLES_GPU:,}')
        print(f'Sampling ~{samples_per_batch_target:,} from each of {n_batches} batches')
        fit_indices = []
        for batch_idx in range(n_batches):
            start_idx = batch_idx * max_samples_per_batch
            end_idx = min((batch_idx + 1) * max_samples_per_batch, n_total)
            batch_size = end_idx - start_idx
            n_sample = min(samples_per_batch_target, batch_size)
            rng = np.random.RandomState(42 + batch_idx)
            chosen = rng.choice(batch_size, n_sample, replace=False) + start_idx
            fit_indices.extend(chosen)
            # Light progress reporting for sampling
            if n_batches <= 10 or (batch_idx + 1) % max(1, n_batches // 10) == 0:
                print(f"   → Sampling progress: batch {batch_idx+1}/{n_batches} (selected ~{len(fit_indices):,} so far)")
        fit_indices = np.array(fit_indices, dtype=np.int64)
        print(f'Step 1: Fitting on {len(fit_indices):,} samples')
        gc.collect()
        emb_fit = emb[fit_indices]
        print('Transferring fit sample to GPU...')
        emb_fit_gpu = cp.asarray(emb_fit, dtype=cp.float32)
        model = cuDBSCAN(eps=float(CHOSEN_EPS), min_samples=int(CHOSEN_MIN_SAMPLES))
        start_time = time.time()
        labels_fit_gpu = model.fit_predict(emb_fit_gpu)
        labels_fit = cp.asnumpy(labels_fit_gpu)
        elapsed = time.time() - start_time
        print(f'Fit completed in {elapsed/60:.2f} minutes — clusters found: {len(set(labels_fit) - {-1})}')
        del labels_fit_gpu, model
        cp.get_default_memory_pool().free_all_blocks()
        # k-NN assignment (GPU)
        remaining_idx = np.setdiff1d(np.arange(n_total), fit_indices)
        labels_full = np.zeros(n_total, dtype=np.int32)
        labels_full[fit_indices] = labels_fit
        from cuml.neighbors import NearestNeighbors as cuNearestNeighbors
        knn = cuNearestNeighbors(n_neighbors=5)
        knn.fit(emb_fit_gpu)
        ASSIGN_BATCH = 500000
        n_assign_batches = (len(remaining_idx) + ASSIGN_BATCH - 1) // ASSIGN_BATCH
        for b in range(n_assign_batches):
            s = b * ASSIGN_BATCH
            e = min((b + 1) * ASSIGN_BATCH, len(remaining_idx))
            idx_batch = remaining_idx[s:e]
            print(f"   → Assigning batch {b+1}/{n_assign_batches} (points {s:,}-{e:,})")
            emb_batch_gpu = cp.asarray(emb[idx_batch], dtype=cp.float32)
            distances, indices = knn.kneighbors(emb_batch_gpu)
            indices_cpu = cp.asnumpy(indices)
            for i_local, global_idx in enumerate(idx_batch):
                neighbor_labels = labels_fit[indices_cpu[i_local]]
                non_noise = neighbor_labels[neighbor_labels != -1]
                if len(non_noise) > 0:
                    labels_full[global_idx] = np.bincount(non_noise).argmax()
                else:
                    labels_full[global_idx] = -1
            del emb_batch_gpu
            cp.get_default_memory_pool().free_all_blocks()
        del emb_fit_gpu, knn, emb_fit, labels_fit
        cp.get_default_memory_pool().free_all_blocks()
        gc.collect()
else:
    # CPU fallback — full or hybrid depending on size
    MAX_FIT_SAMPLES = 10_000_000
    if n_total <= MAX_FIT_SAMPLES:
        print('\nCPU full clustering (may be slow)...')
        start_time = time.time()
        model = skDBSCAN(eps=float(CHOSEN_EPS), min_samples=int(CHOSEN_MIN_SAMPLES), algorithm='ball_tree', n_jobs=-1)
        labels_full = model.fit_predict(emb)
        elapsed = time.time() - start_time
        print(f'CPU DBSCAN completed in {elapsed/60:.2f} minutes')
        del model
    else:
        print('\nCPU hybrid: fit on sample then assign via k-NN')
        rng = np.random.RandomState(42)
        fit_idx = rng.choice(n_total, MAX_FIT_SAMPLES, replace=False)
        emb_fit = emb[np.sort(fit_idx)]
        start_time = time.time()
        model = skDBSCAN(eps=float(CHOSEN_EPS), min_samples=int(CHOSEN_MIN_SAMPLES), algorithm='ball_tree', n_jobs=-1)
        labels_fit = model.fit_predict(emb_fit)
        elapsed = time.time() - start_time
        print(f'Fit completed in {elapsed/60:.2f} minutes — clusters found: {len(set(labels_fit) - {-1})}')
        remaining_idx = np.setdiff1d(np.arange(n_total), fit_idx)
        labels_full = np.zeros(n_total, dtype=np.int32)
        labels_full[fit_idx] = labels_fit
        if len(remaining_idx) > 0:
            knn = skNearestNeighbors(n_neighbors=5, algorithm='ball_tree', n_jobs=-1)
            knn.fit(emb_fit)
            BATCH = 100000
            n_batches = (len(remaining_idx) + BATCH - 1) // BATCH
            for b in range(n_batches):
                s = b * BATCH
                e = min((b + 1) * BATCH, len(remaining_idx))
                idx_batch = remaining_idx[s:e]
                if n_batches <= 10 or (b + 1) % max(1, n_batches // 10) == 0:
                    print(f"   → CPU assignment progress: batch {b+1}/{n_batches} (points {s:,}-{e:,})")
                distances, indices = knn.kneighbors(emb[idx_batch])
                for i_local, global_idx in enumerate(idx_batch):
                    neighbor_labels = labels_fit[indices[i_local]]
                    non_noise = neighbor_labels[neighbor_labels != -1]
                    if len(non_noise) > 0:
                        labels_full[global_idx] = np.bincount(non_noise).argmax()
                    else:
                        labels_full[global_idx] = -1
            del knn
        del model, emb_fit, labels_fit
        gc.collect()
# Save results if labels computed
if labels_full is not None:
    np.save('dbscan_labels.npy', labels_full)
    config_array = np.array([CHOSEN_EPS, CHOSEN_MIN_SAMPLES, 1 if USE_COSINE_DISTANCE else 0, 1 if USE_GPU else 0])
    np.save('dbscan_config.npy', config_array)
    print('Saved: dbscan_labels.npy and dbscan_config.npy')
    n_clusters_final = len(set(labels_full) - {-1})
    n_noise_final = int(np.sum(labels_full == -1))
    print(f'Final clusters: {n_clusters_final}, noise: {n_noise_final} ({n_noise_final/n_total*100:.2f}%)')
else:
    raise RuntimeError('Clustering did not produce labels_full')

gc.collect()


In [ ]:
# Evaluation: cluster counts & metrics (adapted from original notebook)
counts = Counter(labels_full)
print('='*70)
print('CLUSTER ANALYSIS')
print('='*70)
for lbl, cnt in sorted(counts.items()):
    pct = (cnt / len(labels_full)) * 100
    cluster_type = 'NOISE' if lbl == -1 else f'Cluster {lbl}'
    print(f' - {cluster_type:12s}: {cnt:8,} samples ({pct:5.2f}%)')
n_noise = counts.get(-1, 0)
n_clusters = len(set(labels_full) - {-1})
n_clustered = len(labels_full) - n_noise
print('Summary:')
print(f'  Total samples:    {len(labels_full):,}')
print(f'  Clusters found:   {n_clusters}')
print(f'  Clustered points: {n_clustered:,} ({(n_clustered/len(labels_full)*100):.1f}%)')
print(f'  Noise points:     {n_noise:,} ({(n_noise/len(labels_full)*100):.1f}%)')
# Metrics on sample if dataset large
SAMPLE_FOR_METRICS = 300000 if USE_GPU else 15000
if len(set(labels_full) - {-1}) > 1:
    n = emb.shape[0]
    if n > SAMPLE_FOR_METRICS:
        print(f'Computing metrics on sample ({SAMPLE_FOR_METRICS:,})...')
        rng = np.random.RandomState(42)
        idx = rng.choice(n, SAMPLE_FOR_METRICS, replace=False)
        lbls_s = labels_full[idx]
        emb_s = emb[idx]
    else:
        print('Computing metrics on full dataset...')
        emb_s = emb
        lbls_s = labels_full
    try:
        sil = silhouette_score(emb_s, lbls_s)
        print(f'  Silhouette Score: {sil:.4f}')
    except Exception as e:
        print('  Silhouette Score: N/A (', e, ')')
    try:
        dbi = davies_bouldin_score(emb_s, lbls_s)
        print(f'  Davies-Bouldin Index: {dbi:.4f} (lower is better)')
    except Exception as e:
        print('  Davies-Bouldin Index: N/A (', e, ')')
else:
    print('Not enough clusters to compute metrics')
# Optional visualizations (small example)
cluster_labels = [lbl for lbl in labels_full if lbl != -1]
if len(cluster_labels) > 0:
    cluster_counts = Counter(cluster_labels)
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.bar(cluster_counts.keys(), cluster_counts.values())
    plt.xlabel('Cluster ID')
    plt.ylabel('Number of samples')
    plt.title('Cluster Size Distribution')
    plt.subplot(1,2,2)
    sizes = list(cluster_counts.values())
    plt.hist(sizes, bins=min(20, len(sizes)), edgecolor='black')
    plt.xlabel('Cluster size')
    plt.ylabel('Frequency')
    plt.title('Cluster Size Histogram')
    plt.tight_layout()
    plt.show()

print('✅ Evaluation complete')